In [1]:
from workflow_agent import LlmHandler

llm_handler = LlmHandler(
    llm_h_type="ollama",
    llm_h_params={
        "connection_string": "http://localhost:11434",
        "model_name": "gpt-oss:20b" #"llama3.1:latest" # "gemma3:27b"
    }
)

In [2]:
from workflow_agent import create_function_item

from typing import Type
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function
def get_weather(city: str):
    """Get the current weather for a city from weather forcast api."""
    pass

class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

# 2. send_report_email function
def send_report_email(city: str, information: list):
    """Sends a report email with given information points to a city."""
    pass

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

# 3. query_database function
def query_database(topic: str, location: str = None, uid: str = None):
    """Get information from the database with provided filters."""
    pass

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")


# Create structured items for each function
available_functions = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

### 0. Initialize Planner and Adaptor

In [3]:
from workflow_agent import WorkflowPlanner
import logging

wp = WorkflowPlanner(
    llm_h = llm_handler,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

In [4]:
from workflow_agent import WorkflowAdaptor
from workflow_agent import InputCollector
import logging

wa = WorkflowAdaptor(
    llm_h = llm_handler, 
    input_collector_class = InputCollector,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

### 1. Generating simple workflow using available functions (no input or output model)

In [5]:
task_description = """Query database to find information on birds and get latest weather for Berlin, then send an email there."""

workflow = await wp.generate_workflow(
    task_description=task_description)

workflow

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Birds Info',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather', 'content': 'source: get_weather.output.condition'}]}}]

In [6]:
adapted_workflow = await wa.adapt_workflow(
    workflow=workflow)

adapted_workflow

DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'Berlin', '[2].args.city': 'Berlin', '[2].args.information[0].title': 'Birds Info', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': 'source: get_weather.output.condition'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': 'Berlin', '[2].id': '3', '[2].args.city': 'Berlin', '[2].args.information[0].title': 'Birds Info', '[2].args.information[0].content': '1.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': '2.output.condition'}
DEBUG:InputCollector:ic_results : ['literal', 'literal', 'literal', 'li

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Birds Info', 'content': '1.output.info'},
    {'title': 'Weather', 'content': '2.output.condition'}]}}]

### 2. Generating simple workflow using available functions (no output model)

In [7]:
task_description_a = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

workflow_a = await wp.generate_workflow(
    input_model = WfInputs,
    task_description=task_description_a)

workflow_a

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'source: WfInputs.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: WfInputs.output.city',
   'information': [{'title': 'Birds Information',
     'content': 'source: query_database.output.info'}]}}]

In [8]:
adapted_workflow_a = await wa.adapt_workflow(
    workflow=workflow_a, 
    input_model = WfInputs)

adapted_workflow_a

DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'source: WfInputs.output.city', '[2].args.city': 'source: WfInputs.output.city', '[2].args.information[0].title': 'Birds Information', '[2].args.information[0].content': 'source: query_database.output.info'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': '0.output.city', '[2].id': '3', '[2].args.city': '0.output.city', '[2].args.information[0].title': 'Birds Information', '[2].args.information[0].content': '1.output.info'}
DEBUG:InputCollector:ic_results : ['literal', 'reference', 'reference', 'literal', 'reference']


[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Information',
     'content': '1.output.info'}]}}]

### 3. Generating simple workflow using available functions

In [9]:
task_description_b = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather was extracted.")
    information: list[EmailInformationPoint] = Field(default=[], description="Information sent via email.")

workflow_b = await wp.generate_workflow(
    input_model = WfInputs,
    output_model = WfOutputs,
    task_description=task_description_b)

workflow_b

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'source: input_model.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: input_model.output.city',
   'information': [{'title': 'Bird Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Condition',
     'content': 'source: get_weather.output.condition'},
    {'title': 'Temperature',
     'content': 'source: get_weather.output.temperature'}]}}]

In [10]:
adapted_workflow_b = await wa.adapt_workflow(
    workflow=workflow_b, 
    output_model = WfOutputs,
    input_model = WfInputs)

adapted_workflow_b

DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:{
  "city": "0.output.city",
  "information": [
    {
      "title": "Bird Information",
      "content": "1.output.info"
    },
    {
      "title": "Weather Condition",
      "content": "2.output.condition"
    },
    {
      "title": "Temperature",
      "content": "2.output.temperature"
    }
  ]
}
 Mapping for key 'content' is invalid.
 Mapping for array item at index 2 is invalid.
 Mapping for key 'information' is invalid.
DEBUG:WorkflowAdaptor:{
  "city": "0.output.city",
  "information": [
    {
      "title": "Bird Information",
      "content": "1.output.info"
    },
    {
      "title": "Weather Condition",
      "content": "2.output.condition"
    },
    {
      "title": "Temperature",
      "content": "2.output.temperature"
    }
  ]
}
 Mapping

IndexError: list index out of range

In [ ]:
break

In [ ]:
"""
This module contains methods to clean up llm generated and adapted workflows 
by turning all literal inputs produced by llm into a populated input model for
the workflow.
"""

import attrs
import attrsx

import re
from typing import Any
import copy


@attrsx.define()
class InputCollector:

    def _extract_leaf_paths(self, data: any, prefix: str = "") -> dict:
        """
        Recursively traverse a nested structure (dicts and lists) and return a dictionary
        mapping each leaf's full path to its value, but skipping any leaves whose key is "name".

        For example, given:
        {
        'name': 'get_weather',
        'args': {'city': 'Berlin'}
        }
        It returns:
        {
        "args.city": "Berlin"
        }
        
        For lists, the index is included in square brackets. For example:
        {
        'information': [
            {'title': 'Weather Forecast', 'content': 'source: get_weather.output.condition'}
        ]
        }
        Would yield:
        {
        "information[0].title": "Weather Forecast",
        "information[0].content": "source: get_weather.output.condition"
        }
        """
        leaves = {}
        
        if isinstance(data, dict):
            for key, value in data.items():
                # Skip the key if it is "name"
                if key == "name":
                    continue
                new_prefix = f"{prefix}.{key}" if prefix else key
                leaves.update(self._extract_leaf_paths(value, new_prefix))
        elif isinstance(data, list):
            for idx, item in enumerate(data):
                new_prefix = f"{prefix}[{idx}]"
                leaves.update(self._extract_leaf_paths(item, new_prefix))
        else:
            # If the final key is "name", skip adding it.
            if not prefix.endswith(".name") and prefix != "name":
                leaves[prefix] = data
            
        return leaves

    def _set_by_path(self, data: any, path: str, value: any) -> any:
        """
        Return a modified copy of a nested structure (dicts/lists) given a path string.
        
        The path format uses dots to separate keys and [index] for list indices.
        For example: "[2].args.information[1].content"
        """
        new_data = copy.deepcopy(data)
        parts = re.split(r'\.(?![^\[]*\])', path)
        current = new_data
        for i, part in enumerate(parts):
            # Extract any list indices at the beginning of the part.
            while part.startswith('['):
                m = re.match(r'\[(\d+)\](.*)', part)
                if m:
                    idx = int(m.group(1))
                    current = current[idx]
                    part = m.group(2)
                else:
                    break
            # When at the last part, perform the update.
            if i == len(parts) - 1:
                if part:
                    # If part contains an index like key[index]
                    m = re.match(r'([^\[]+)(\[(\d+)\])?', part)
                    if m:
                        key = m.group(1)
                        if m.group(2):
                            idx = int(m.group(3))
                            current[key][idx] = value
                        else:
                            current[key] = value
                else:
                    # If no part remains, update current directly.
                    current = value
            else:
                if part:
                    m = re.match(r'([^\[]+)(\[(\d+)\])?', part)
                    if m:
                        key = m.group(1)
                        current = current[key]
                        if m.group(2):
                            idx = int(m.group(3))
                            current = current[idx]
                else:
                    continue
        return new_data

    def _update_workflow_from_leaves(self, workflow: Any, leaf_paths: list, new_values: list) -> Any:
        """
        Update the workflow based on a list of leaf paths,
        and a list of new values. For each leaf path, if the corresponding decision is True, the leaf's value
        in the workflow is replaced with the new value.
        """
        for path, new_value in zip(leaf_paths, new_values):
            workflow = self._set_by_path(workflow, path, new_value)
        return workflow

    def _classify_value(self, value: str) -> str:
        """
        Classify the input value as either 'reference' or 'literal' based on regex.
        If the value contains 'source:' or '.output.', it is classified as a reference.
        Otherwise, it is a literal.
        """
        # Pattern matches if "source:" or ".output." appears anywhere in the string.
        print(f"value type: {type(value)}")
        if (value is not None) and (re.search(r"(source:|\.output\.)", str(value))):
            return "reference"
        return "literal"

    def fix_literal_values(self, planned_workflow : dict, adapted_workflow : dict):

        """
        Replaces inputs that suppose to be literal in llm adapted workflow
        based on planned workflow.
        """

        og_leaves = self._extract_leaf_paths(planned_workflow)
        for k,v in og_leaves.items():
            if v is not None:
                v = str(v)
            og_leaves[k] = v

        mod_leaves = self._extract_leaf_paths(adapted_workflow)
        for k,v in mod_leaves.items():
            if v is not None:
                v = str(v)
            mod_leaves[k] = v

        self.logger.debug(f"og_leaves : {og_leaves}")
        self.logger.debug(f"mod_leaves : {mod_leaves}")


        ic_results = [self._classify_value(value) for value in og_leaves.values()]

        self.logger.debug(f"ic_results : {ic_results}")

        new_values = [mod_leaves[[olk for olk in list(mod_leaves.keys()) \
            if inputstr in olk][0]] if cl == 'reference' else og_leaves[inputstr] \
                for inputstr, cl in zip(og_leaves, ic_results)]

        leaf_paths = list(og_leaves.keys())

        cor_workflow = self._update_workflow_from_leaves(adapted_workflow, leaf_paths, new_values)

        return cor_workflow

    

In [ ]:
ic = InputCollector(loggerLvl=logging.DEBUG)

In [ ]:
ic.fix_literal_values(workflow_a, adapted_workflow_a)

DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'source: input_model.output.city', '[2].args.city': 'source: input_model.output.city', '[2].args.information[0].title': 'Birds Information', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather Update', '[2].args.information[1].content': 'source: get_weather.output.condition'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': '0.output.city', '[2].id': '3', '[2].args.city': '0.output.city', '[2].args.information[0].title': 'Birds Information', '[2].args.information[0].content': '1.output.info', '[2].args.information[1].title': 'Weather Update', '[2].args.information[1].content': '2.output.condition'}
DEBUG:InputCollector:ic_results : ['literal', 'reference', 'reference', 'literal', 'reference', 'literal', 'reference']


value type: <class 'str'>
value type: <class 'str'>
value type: <class 'str'>
value type: <class 'str'>
value type: <class 'str'>
value type: <class 'str'>
value type: <class 'str'>


[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Information', 'content': '1.output.info'},
    {'title': 'Weather Update', 'content': '2.output.condition'}]}}]

In [ ]:
wa.input_collector_h.fix_literal_values(workflow_b, adapted_workflow)

DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'source: input_model.output.city', '[2].args.city': 'source: input_model.output.city', '[2].args.information[0].title': 'Birds Information', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather Update', '[2].args.information[1].content': 'source: get_weather.output.condition', '[3].id': 4}
DEBUG:InputCollector:mod_leaves : {'[0].id': 1, '[0].args.topic': 'birds', '[1].id': 2, '[1].args.city': 'Berlin', '[2].id': 3, '[2].args.city': 'Berlin', '[2].args.information[0].title': 'Birds', '[2].args.information[0].content': '1.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': '2.output.condition'}


TypeError: expected string or bytes-like object